# # 01 - Exploratory Data Analysis
# Dataset: Fragrantica Perfumes
# Target: longevity_avg (skala 1-5)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA = ROOT / "data" / "perfumes.csv"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
print("ROOT :", ROOT)
print("DATA :", DATA.exists())

ROOT : C:\Users\ThinkPad\Documents\Semester 5\Machine Learning\Project Akhir - Prediksi Longevity Parfum
DATA : True


In [2]:
df = pd.read_csv(DATA)
print("Baris :", df.shape[0])
print("Kolom :", df.shape[1])

Baris : 140228
Kolom : 59


In [3]:
print("Daftar kolom:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:2d}. {c}")

Daftar kolom:
 1. id
 2. slug
 3. name
 4. brand
 5. year
 6. collection
 7. gender
 8. url
 9. rating_avg
10. vote_count
11. rating_b1
12. rating_b2
13. rating_b3
14. rating_b4
15. rating_b5
16. longevity_avg
17. longevity_b1
18. longevity_b2
19. longevity_b3
20. longevity_b4
21. longevity_b5
22. sillage_avg
23. sillage_b1
24. sillage_b2
25. sillage_b3
26. sillage_b4
27. price_value_avg
28. price_value_b1
29. price_value_b2
30. price_value_b3
31. price_value_b4
32. price_value_b5
33. have
34. had
35. want
36. perceived_female
37. perceived_female_leaning
38. perceived_unisex
39. perceived_male_leaning
40. perceived_male
41. winter
42. spring
43. summer
44. autumn
45. day
46. night
47. people
48. magnitude
49. compound_magnitude
50. recent_magnitude
51. last_comment_at
52. scraped_at
53. accords
54. notes_top
55. notes_middle
56. notes_base
57. notes_flat
58. perfumers
59. description


In [4]:
print("Tipe data:")
print(df.dtypes.to_string())

Tipe data:
id                            int64
slug                            str
name                            str
brand                           str
year                        float64
collection                      str
gender                          str
url                             str
rating_avg                  float64
vote_count                    int64
rating_b1                     int64
rating_b2                     int64
rating_b3                     int64
rating_b4                     int64
rating_b5                     int64
longevity_avg               float64
longevity_b1                  int64
longevity_b2                  int64
longevity_b3                  int64
longevity_b4                  int64
longevity_b5                  int64
sillage_avg                 float64
sillage_b1                    int64
sillage_b2                    int64
sillage_b3                    int64
sillage_b4                    int64
price_value_avg             float64
price_value_b1   

In [5]:
# Nilai hilang per kolom, hanya yang ada isinya
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
tbl_miss = pd.DataFrame({
    "jumlah_hilang": miss,
    "persen": (miss / len(df) * 100).round(2),
})
print(tbl_miss.to_string())
tbl_miss.to_csv(OUT / "tabel_missing.csv")

              jumlah_hilang  persen
notes_flat           101591   72.45
perfumers             87211   62.19
collection            81238   57.93
notes_middle          41517   29.61
notes_base            41467   29.57
notes_top             41417   29.54
year                  25267   18.02
people                 3116    2.22
accords                2791    1.99


In [6]:
print("Baris duplikat (semua kolom):", df.duplicated().sum())
print("ID duplikat                 :", df["id"].duplicated().sum())
print("slug duplikat               :", df["slug"].duplicated().sum())

Baris duplikat (semua kolom): 0
ID duplikat                 : 0
slug duplikat               : 1563


In [7]:
print("Statistik target longevity_avg:")
print(df["longevity_avg"].describe().round(3).to_string())

Statistik target longevity_avg:
count    140228.000
mean          2.604
std           1.422
min           0.000
25%           2.162
50%           3.000
75%           3.546
max           5.000


In [8]:
# Cek nilai 0 pada target, itu bukan nilai asli
nol = df["longevity_avg"] == 0
print("Baris longevity = 0        :", int(nol.sum()))
print("Dari situ vote_count = 0   :", int((nol & (df["vote_count"] == 0)).sum()))
print("Persen dari seluruh data   :", round(nol.mean() * 100, 2), "%")
print("Nilai minimum selain 0     :", round(df.loc[~nol, "longevity_avg"].min(), 2))

Baris longevity = 0        : 26777
Dari situ vote_count = 0   : 17540
Persen dari seluruh data   : 19.1 %
Nilai minimum selain 0     : 1.0


In [9]:
# Tabel ambang vote_count, dasar pemilihan ambang 500
baris = []
for amb in [0, 20, 100, 500, 1000, 2000, 5000]:
    sub = df[(df["vote_count"] >= amb) & (df["longevity_avg"].notna())]
    baris.append({
        "ambang_vote": amb,
        "jumlah_baris": len(sub),
        "persen_data": round(len(sub) / len(df) * 100, 2),
        "std_target": round(sub["longevity_avg"].std(), 3),
        "mean_target": round(sub["longevity_avg"].mean(), 3),
    })
tbl_vote = pd.DataFrame(baris)
print(tbl_vote.to_string(index=False))
tbl_vote.to_csv(OUT / "tabel_ambang_vote.csv", index=False)

 ambang_vote  jumlah_baris  persen_data  std_target  mean_target
           0        140228       100.00       1.422        2.604
          20         57273        40.84       0.532        3.186
         100         26313        18.76       0.486        3.205
         500          8211         5.86       0.460        3.271
        1000          4341         3.10       0.446        3.312
        2000          2046         1.46       0.435        3.362
        5000           631         0.45       0.439        3.415


In [10]:
# Histogram target pada data terfilter
sub = df[(df["vote_count"] >= 500) & (df["longevity_avg"].notna())]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sub["longevity_avg"], bins=30, color="#2F6F8F", edgecolor="white")
ax.set_title("Sebaran longevity_avg (vote_count >= 500)")
ax.set_xlabel("longevity_avg (1-5)")
ax.set_ylabel("Jumlah parfum")
fig.tight_layout()
fig.savefig(OUT / "fig_hist_longevity.png", dpi=150)
plt.close(fig)
print("Tersimpan:", OUT / "fig_hist_longevity.png")
print("Skewness:", round(sub["longevity_avg"].skew(), 3))

Tersimpan: C:\Users\ThinkPad\Documents\Semester 5\Machine Learning\Project Akhir - Prediksi Longevity Parfum\output\fig_hist_longevity.png
Skewness: 0.002


In [11]:
# Korelasi antar kolom numerik utama, dihitung pada data terfilter
num = ["longevity_avg", "sillage_avg", "rating_avg", "price_value_avg",
       "vote_count", "year", "have", "want", "winter", "summer"]
koor = sub[num].corr(numeric_only=True).round(3)
print(koor.to_string())
koor.to_csv(OUT / "tabel_korelasi.csv")

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(koor.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(num))); ax.set_yticklabels(num, fontsize=8)
for i in range(len(num)):
    for j in range(len(num)):
        ax.text(j, i, koor.values[i, j], ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(OUT / "fig_korelasi.png", dpi=150)
plt.close(fig)
print("Tersimpan:", OUT / "fig_korelasi.png")

                 longevity_avg  sillage_avg  rating_avg  price_value_avg  vote_count   year   have   want  winter  summer
longevity_avg            1.000        0.903       0.293           -0.216       0.114  0.044  0.116  0.219   0.239  -0.047
sillage_avg              0.903        1.000       0.206           -0.132       0.070 -0.082  0.051  0.112   0.182  -0.094
rating_avg               0.293        0.206       1.000           -0.037       0.086 -0.011  0.156  0.244   0.177   0.090
price_value_avg         -0.216       -0.132      -0.037            1.000      -0.053 -0.132 -0.001 -0.184   0.001   0.000
vote_count               0.114        0.070       0.086           -0.053       1.000 -0.071  0.951  0.820   0.816   0.727
year                     0.044       -0.082      -0.011           -0.132      -0.071  1.000  0.002  0.075   0.010  -0.001
have                     0.116        0.051       0.156           -0.001       0.951  0.002  1.000  0.864   0.854   0.757
want                    

Tersimpan: C:\Users\ThinkPad\Documents\Semester 5\Machine Learning\Project Akhir - Prediksi Longevity Parfum\output\fig_korelasi.png


In [12]:
# Bentuk data teks yang harus diubah jadi angka
print("Contoh accords   :", df["accords"].dropna().iloc[0])
print("Contoh notes_top :", df["notes_top"].dropna().iloc[0])
print("Contoh notes_base:", df["notes_base"].dropna().iloc[0])

Contoh accords   : citrus:100|fresh spicy:47|green:41|sweet:33|aromatic:31|fruity:27|white floral:26|honey:16
Contoh notes_top : Orange:100|Bergamot:28|Basil:25|Grapefruit:23|Green Notes:22|Mint:15|Lemon:12
Contoh notes_base: Honey:25|Amber:20|Cedar:12|Musk:10


In [13]:
print("Jumlah jenis accord unik :", df["accords"].dropna().str.split("|").explode().str.split(":").str[0].nunique())
print("Jumlah parfum tanpa accords :", int(df["accords"].isna().sum()))
print("Jumlah parfum tanpa notes_base :", int(df["notes_base"].isna().sum()))

Jumlah jenis accord unik : 92
Jumlah parfum tanpa accords : 2791
Jumlah parfum tanpa notes_base : 41467


In [14]:
# Kesimpulan EDA
print("RINGKASAN")
print("-" * 50)
print(f"Data mentah            : {len(df):,} baris")
print(f"Tanpa vote             : {int((df['vote_count'] == 0).sum()):,} baris")
print(f"Target bernilai 0 palsu: {int(nol.sum()):,} baris")
print(f"Setelah filter >= 500  : {len(sub):,} baris")
print(f"std target sesudahnya  : {sub['longevity_avg'].std():.3f}")
print(f"Korelasi sillage       : {koor.loc['longevity_avg', 'sillage_avg']:.3f} (bocor)")

RINGKASAN
--------------------------------------------------
Data mentah            : 140,228 baris
Tanpa vote             : 18,996 baris
Target bernilai 0 palsu: 26,777 baris
Setelah filter >= 500  : 8,211 baris
std target sesudahnya  : 0.460
Korelasi sillage       : 0.903 (bocor)
